# data processing
refactored to df

## imports

In [20]:
%reload_ext autoreload
%autoreload 2

In [21]:
import sys
import os
from pathlib import Path
import configparser
config = configparser.ConfigParser()
config.read_file(open('privateconfig'))
resdir = Path(config['Datafolder']['data'])
workdir = Path(config['Codefolder']['workspace'])
os.chdir(workdir)

In [22]:
# analysis
from scipy.io import loadmat
from sklearn.decomposition import FastICA
from sklearn.datasets import make_regression
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoCV, Lasso
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr


In [23]:
# misc
import pickle
from collections import defaultdict

In [24]:
# task
from env_config import Config
from firefly_task import ffacc_real
# from monkey_functions import *
# from InverseFuncs import *
from stable_baselines3 import TD3
import torch


In [25]:
from neural_plot_ult import *
import time
tic=time.time()
import warnings
warnings.filterwarnings('ignore')

In [26]:
# const
bin_size = 1  # how many bin of DT
# num_bins = 24  # how many bins to use. use 2.4 s and discard the long trials.
# monkey_height = 10
DT = 0.006  # DT for raw data
# reward_boundary = 65
areas = ['PPC', 'PFC', 'MST']
worldscale = 200


m = 'm51'
folder = 'm51_mat_ruiyi'
dens = [0.0001, 0.0005, 0.001,  0.005]

locals().update({m: {}})
figure_path = resdir/'figures'
# datapaths = [i for i in Pa1th(resdir/'mat_ruiyi').glob(f'{m}*.mat')]
datapaths = [i for i in Path(resdir/folder).glob(f'{m}*.mat')]
session=[int(a.stem[-2:]) for a in datapaths]


## from raw data file: task relavent variables and neural

In [27]:
# load raw data
df=pd.DataFrame() 
for idx, datapath in enumerate(datapaths):
    if datapath.stem[-1].isalpha():
        continue
    data = loadmat(datapath)
    eval(m)[datapath.stem] = data

    # df
    sessdf=pd.DataFrame() 
    sessdf['trial']=np.arange((len(data['trials_behv'][0])))
    sessdf['session']=int(datapath.stem[(datapath.stem).find('s')+1:])
    df=pd.concat([df, sessdf])

In [28]:
# testing iti df
itidata = defaultdict(list)

for key, data in eval(m).items():
    sess = int(key[-2:])
    if key[-1].isalpha():
        continue

    trials_behv = data['trials_behv'][0]
    trials_units = data['units'][0]
    units_area = np.array([v[0] for v in trials_units['brain_area']])
    trials_error = []
    trials_error_sign = []
    trials_target_angle = []
    trials_target_distance = []

    for trial_idx, trial_behv in enumerate(trials_behv):
        trial_ts = trial_behv['continuous']['ts'][0][0].reshape(-1)
        t_mask = (trial_ts > 0) & (
            ~np.isnan(trial_behv['continuous']['ymp'][0][0].reshape(-1)))
        t_mask &= trial_ts > trial_behv['events']['t_stop'][0][0].reshape(-1)
        if t_mask.sum() > 0:
            # remove the first data point to avoid downsample error
            t_mask[np.where(t_mask == True)[0][0]] = False

        # task varaibles from data
        mx = trial_behv['continuous']['xmp'][0][0][t_mask]
        my = trial_behv['continuous']['ymp'][0][0][t_mask]
        fx = trial_behv['continuous']['xfp'][0][0][t_mask]
        fy = trial_behv['continuous']['yfp'][0][0][t_mask]
        eye_hor_theta = trial_behv['continuous']['yre'][0][0][t_mask] # use yle for left eye.
        eye_ver_theta = trial_behv['continuous']['zre'][0][0][t_mask]
        # print(len(eye_hor_theta), len(eye_ver_theta), len(mx))
        mv = trial_behv['continuous']['v'][0][0][t_mask].reshape(-1, 1)
        mw = trial_behv['continuous']['w'][0][0][t_mask].reshape(-1, 1)

        # some adjustment for screen distance
        sx = np.ones_like(fx)
        sy = np.ones_like(fy)
        if my.size > 0:
            fx = np.ones_like(fx) * fx[0]
            fy = np.ones_like(fy) * fy[0]
            sx *= mx[-1]
            sy *= my[-1]
            my = my + 30
            fy = fy + 30
            sy = sy + 30

        # some thing coudl be removed from there.
        dx = fx - mx; dy = fy - my
        rel_dist = np.sqrt(dx**2 + dy**2); rel_ang = np.rad2deg(np.arctan2(dy, dx))
        rel_dist_stop = np.sqrt((sx - mx)**2 + (sy - my)**2)
        abs_dist = np.sqrt(mx**2 + my**2); abs_ang = np.rad2deg(np.arctan2(my, mx))
        body_theta = np.deg2rad(np.cumsum(mw*-1) * DT + 90)
        body_x, body_y = mx.reshape(-1), my.reshape(-1)


        # # skip bad trial
        # if t_mask.sum() * DT > 3.5 or t_mask.sum() * DT < 0.6 or mv.max() < 50 or \
        #         abs_dist[-1] < np.sqrt(fx**2 + fy**2)[-1] * 0.3:
        #     continue

        # errors
        if my.size > 0:
            trials_error.append(rel_dist[-1][0])
            trials_error_sign.append(rel_dist[-1][0])
            trials_target_angle.append(
                np.rad2deg(np.arctan2(fy, fx))[-1][0] - 90)
            trials_target_distance.append(np.sqrt(fx**2 + fy**2)[-1][0])
            d1 = np.sqrt(fx**2 + fy**2)
            r1 = (fx**2 + fy**2) / (2*fx)
            radian1 = 2 * r1 * np.arcsin(d1 / (2 * r1))
            d2 = np.sqrt(mx**2 + my**2)
            r2 = (mx**2 + my**2) / (2*mx + 1e-8)
            radian2 = 2 * r2 * np.arcsin(d2 / (2 * r2 + 1e-8))
            sign = np.ones_like(rel_dist)
            sign[radian2 < radian1] = -1
            rel_dist = sign * rel_dist
            trials_error_sign[-1] = rel_dist[-1][0]
        else:
            trials_error.append(np.nan)
            trials_error_sign.append(np.nan)
            trials_target_angle.append(np.nan)
            trials_target_distance.append(np.nan)

        target_variable = np.hstack([rel_dist, rel_ang, abs_dist, abs_ang,
                                     eye_hor_theta, eye_ver_theta, 
                                     fx, fy, mx, my, mv, mw])

        target_variable_ds = downsample(target_variable, bin_size=bin_size)

        (rel_dist, rel_ang, abs_dist, abs_ang,
         eye_hor_theta, eye_ver_theta,
         fx, fy, mx, my, mv, mw) = zip(*target_variable_ds)
        body_theta = -np.deg2rad(np.cumsum(mw) * 0.1/17 - 90)
        body_x, body_y = np.array(mx).reshape(-1), np.array(my).reshape(-1)
        latent_ff_hori, latent_ff_vert = convert_location_to_angle(abs(np.array(rel_dist)).reshape(-1), np.array(fx).reshape(-1), np.array(fy).reshape(-1),
                                                                   np.array(body_theta), np.array(
                                                                       body_x), np.array(body_y),
                                                                   np.array(eye_hor_theta).reshape(-1), np.array(eye_ver_theta).reshape(-1), remove_pre=False)

        # df
        itidata['session'].append(int(sess))
        itidata['trial'].append(trial_idx)
        itidata['fullon'].append(trial_behv['logical']['firefly_fullON'][0][0][0][0])
        itidata['density'].append(
            (trial_behv['prs'][0][0]['floordensity'].item()))
        # itidata['rel_dist'].append(rel_dist)
        # itidata['rel_ang'].append(rel_ang)
        # itidata['abs_dist'].append(abs_dist)
        # itidata['abs_ang'].append(abs_ang)
        itidata['eye_hori'].append(eye_hor_theta)
        itidata['eye_vert'].append(eye_ver_theta)
        itidata['ff_hori'].append(latent_ff_hori.reshape(-1))
        itidata['ff_vert'].append(latent_ff_vert.reshape(-1))
        itidata['fx'].append(fx)
        itidata['fy'].append(fy)
        itidata['mx'].append(mx)
        itidata['my'].append(my)
        itidata['mv'].append(mv)
        itidata['mw'].append(mw)

        # neural
        activities = []  # activities for all neurons for 1 trial. shape: ts, neurons
        for trials_unit in trials_units:
            fire_ts = trials_unit['trials'][0][trial_idx][0].reshape(-1)
            if fire_ts.size > 0 and fire_ts[-1] >= trial_ts[-1]:
                fire_ts = fire_ts[:-1]
            activity = np.zeros_like(trial_ts)
            bin_indices = np.digitize(fire_ts, trial_ts)
            unique_bins, bin_counts = np.unique(
                bin_indices, return_counts=True)
            activity[unique_bins] = bin_counts
            activities.append(activity)

        activities = np.vstack(activities).T   # time * unit
        activities = activities[t_mask]
        activities = gaussian_filter1d(
            activities, sigma=4, axis=0)  # neural for each trial
        activityds = downsample(activities, bin_size=bin_size)
        activity_var_ds = downsample_variance(activities, bin_size=bin_size)
        
        for area in areas:
            area_mask = [v in area for v in units_area]
            if sum(area_mask) == 0:
                activity_ = np.nan
            else:
                activity_ = activityds[:, area_mask]  # area activity
                activity_var_ds_=activity_var_ds[:,area_mask]
            itidata[area].append(activity_)
            itidata[f'{area}_var'].append(activity_var_ds_)

    itidata['error'] += (trials_error)
    itidata['error_sign'] += (trials_error_sign)
    itidata['target_angle'] += (trials_target_angle)
    itidata['target_distance'] += (trials_target_distance)

itidf = pd.DataFrame(itidata)
# itidf = pd.merge(df, tmp, on=['trial', 'session'], how='inner')
print(len(itidf))
df = pd.merge(df, itidf, on=['trial', 'session'], how='inner')

6599


In [29]:
df['timer']=df.apply(lambda x: np.arange(len(x.mx)), axis=1)
df['countdown']=df.apply(lambda x: np.flip(-np.arange(len(x.mx)), axis=0), axis=1)

In [30]:
df.columns

Index(['trial', 'session', 'fullon', 'density', 'eye_hori', 'eye_vert',
       'ff_hori', 'ff_vert', 'fx', 'fy', 'mx', 'my', 'mv', 'mw', 'PPC',
       'PPC_var', 'PFC', 'PFC_var', 'MST', 'MST_var', 'error', 'error_sign',
       'target_angle', 'target_distance', 'timer', 'countdown'],
      dtype='object')

In [31]:
# check len
random_rows = df.sample(n=3)

# Iterate over the columns
for col in random_rows.columns:
    try:
        # Get the column values for the selected rows
        col_values = random_rows[col]
        # Calculate the length of each column value
        lengths = col_values.apply(lambda x: len(x))
        # Print column name and the length of each column value
        print(f"Column: {col}, Lengths: {lengths.tolist()}")
    except: continue


Column: eye_hori, Lengths: [245, 217, 190]
Column: eye_vert, Lengths: [245, 217, 190]
Column: ff_hori, Lengths: [245, 217, 190]
Column: ff_vert, Lengths: [245, 217, 190]
Column: fx, Lengths: [245, 217, 190]
Column: fy, Lengths: [245, 217, 190]
Column: mx, Lengths: [245, 217, 190]
Column: my, Lengths: [245, 217, 190]
Column: mv, Lengths: [245, 217, 190]
Column: mw, Lengths: [245, 217, 190]
Column: PPC, Lengths: [245, 217, 190]
Column: PPC_var, Lengths: [245, 217, 190]
Column: PFC_var, Lengths: [245, 217, 190]
Column: MST_var, Lengths: [245, 217, 190]
Column: timer, Lengths: [245, 217, 190]
Column: countdown, Lengths: [245, 217, 190]


In [32]:
# check size
random_rows = df.sample(n=3)

# Iterate over the columns
for col in random_rows.columns:
    try:
        # Get the column values for the selected rows
        col_values = random_rows[col]
        # Calculate the length of each column value
        lengths = col_values.apply(lambda x: (x.shape))
        # Print column name and the length of each column value
        print(f"Column: {col}, Lengths: {lengths.tolist()}")
    except: continue

Column: ff_hori, Lengths: [(84,), (221,), (197,)]
Column: ff_vert, Lengths: [(84,), (221,), (197,)]
Column: PPC, Lengths: [(84, 86), (221, 112), (197, 112)]
Column: PPC_var, Lengths: [(84, 86), (221, 112), (197, 112)]
Column: PFC_var, Lengths: [(84, 86), (221, 112), (197, 112)]
Column: MST_var, Lengths: [(84, 86), (221, 112), (197, 112)]
Column: timer, Lengths: [(84,), (221,), (197,)]
Column: countdown, Lengths: [(84,), (221,), (197,)]


## the varialbes we have:

In [33]:
# with neuron variance as PPC_var
date='0421iti'
df.to_pickle(resdir/f'{date}_m51df.pkl')
notify(f'saved: {date}_m51df.pkl')
print(f'saved: {date}_m51df.pkl')

saved: 0421iti_m51df.pkl
